# Transformación de datos

**Autor:** Jazna Meza Hidalgo

**Correo Electrónico:** ja.meza@profesor.duoc.cl

**Fecha de Creación:** Marzo de 2026  
**Versión:** 1.0  

---

## Descripción

Este notebook ofrece la explicación de la transformación de datos.

Mantiene el uso de pipeline como buena práctica de la industria.


---

## Requisitos de Software

Este notebook fue desarrollado con Python 3.12. A continuación se listan las bibliotecas necesarias:

- pandas (>=1.1.0)
- matplotlib (3.7.1)
- numpy (2.0.2)


Para verificar la versión instalada ejecutar usando el siguiente comando, usando la librería de la cual quieres saber la versión:

```python
import pandas as pd
print(pd.__version__)
````

In [ ]:
!wget https://raw.githubusercontent.com/JaznaLaProfe/Programacion_para_ciencia_de_datos/master/data/data_clientes_gastos.csv

--2026-03-25 20:16:52--  https://raw.githubusercontent.com/JaznaLaProfe/Programacion_para_ciencia_de_datos/master/data/data_clientes_gastos.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 22634 (22K) [text/plain]
Saving to: ‘data_clientes_gastos.csv’

data_clientes_gasto 100%[===================>]  22.10K  --.-KB/s    in 0.001s  

2026-03-25 20:16:53 (16.9 MB/s) - ‘data_clientes_gastos.csv’ saved [22634/22634]



In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler, OrdinalEncoder

In [ ]:
# Carga el set de datos
data = pd.read_csv('data_clientes_gastos.csv')

## Revisión del estado

In [ ]:
# Cantidad de observaciones y columnas
data.shape

(1000, 4)

## Valores faltantes y tipos de datos

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   edad             1000 non-null   int64 
 1   ingreso_mensual  1000 non-null   int64 
 2   gasto_mensual    1000 non-null   int64 
 3   tipo_cliente     1000 non-null   object
dtypes: int64(3), object(1)
memory usage: 31.4+ KB


In [ ]:
# Muestra las medidas estadísticas
data.describe()

,edad,ingreso_mensual,gasto_mensual
count,1000.00000,1.000000e+03,1000.000000
mean,44.08000,8.071271e+05,301216.437000
std,14.61017,2.469581e+05,99927.772676
min,18.00000,2.000000e+05,50000.000000
25%,32.00000,6.414525e+05,232958.750000
50%,44.00000,8.059435e+05,295358.500000
75%,57.00000,9.745115e+05,365499.000000
max,69.00000,1.540952e+06,591291.000000


In [ ]:
data.describe(include="object")

,tipo_cliente
count,1000
unique,3
top,bajo
freq,400


In [ ]:
data.tipo_cliente.unique()

array(['bajo', 'alto', 'medio'], dtype=object)

# Ingeniería de características `Feature Engineering`

In [ ]:
def agregar_variables(data : pd.DataFrame):
  """
  Agrega variables al set de datos.
  Parámetros:
    data (pd.DataFrame): Set de datos.
  Retorna:
    pd.DataFrame
  """
  data = data.copy()

  # Ratio gasto / ingreso
  data["ratio_gasto"] = data["gasto_mensual"] / data["ingreso_mensual"]

  # Clasificación simple
  data["nivel_gasto"] = np.where(data["ratio_gasto"] > 0.5, "alto", "bajo")

  return data

In [ ]:
# Crear nuevas variables
data = agregar_variables(data)

In [ ]:
data.head()

,edad,ingreso_mensual,gasto_mensual,tipo_cliente,ratio_gasto,nivel_gasto
0,39,897655,196443,bajo,0.218840,bajo
1,69,1028935,220327,bajo,0.214131,bajo
2,62,527749,328904,bajo,0.623221,alto
3,52,1012258,226485,bajo,0.223742,bajo
4,58,1084493,187893,alto,0.173254,bajo


In [ ]:
data.nivel_gasto.value_counts()

,count
nivel_gasto,
bajo,723
alto,277


In [ ]:
data.tipo_cliente.value_counts()

,count
tipo_cliente,
bajo,400
medio,396
alto,204


# Transformación (escalado y codificación)

In [ ]:
# Define las variables cualitativas y cuantitativas disponibles
numeric_features = ["edad", "ingreso_mensual", "gasto_mensual", "ratio_gasto"]
categorical_features = ["tipo_cliente", "nivel_gasto"]

## Uso de `StandardScaler`y `OneHotEncoder`

In [ ]:
# Define el pipeline para las variables cuantitativas
pipeline_numerico = Pipeline(
    steps=[
        ("escalado", StandardScaler())
    ]
)

In [ ]:
# Define el pipeline para las variables cualitativas
pipeline_categorico = Pipeline(
    steps=[
        # Acá está lo que había quedado pendiente la semana anterior
        ("encoder", OneHotEncoder(sparse_output=False, handle_unknown="ignore"))
    ]
)

In [ ]:
# Integra ambos pipelines
preprocesador = ColumnTransformer(
    transformers=[
        ("num", pipeline_numerico, numeric_features),
        ("cat", pipeline_categorico, categorical_features)
    ]
)

In [ ]:
# Define el pipeline de transformación
pipeline_transformacion = Pipeline(
    steps=[
        ("transformacion", preprocesador)
    ]
)

In [ ]:
# Aplica la transformación
data_transformada = pipeline_transformacion.fit_transform(data)

In [ ]:
# Crea el Dataframe con los datos limpios
data_transformada = pd.DataFrame(
    pipeline_transformacion.fit_transform(data),
    columns=pipeline_transformacion.named_steps["transformacion"].get_feature_names_out()
)

# Setea el nombre de las columnas
data_transformada.columns = data_transformada.columns.str.replace("num__", "")
data_transformada.columns = data_transformada.columns.str.replace("cat__", "")
data_transformada[numeric_features] = data_transformada[numeric_features].apply(pd.to_numeric)

In [ ]:
data_transformada.head()

,edad,ingreso_mensual,gasto_mensual,ratio_gasto,tipo_cliente_alto,tipo_cliente_bajo,tipo_cliente_medio,nivel_gasto_alto,nivel_gasto_bajo
0,-0.347877,0.366755,-1.049016,-0.849274,0.0,1.0,0.0,0.0,1.0
1,1.706515,0.898610,-0.809884,-0.868914,0.0,1.0,0.0,0.0,1.0
2,1.227157,-1.131843,0.277214,0.837218,0.0,1.0,0.0,1.0,0.0
3,0.542359,0.831046,-0.748229,-0.828829,0.0,1.0,0.0,0.0,1.0
4,0.953238,1.123692,-1.134621,-1.039393,1.0,0.0,0.0,0.0,1.0


In [ ]:
data_transformada.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   edad                1000 non-null   float64
 1   ingreso_mensual     1000 non-null   float64
 2   gasto_mensual       1000 non-null   float64
 3   ratio_gasto         1000 non-null   float64
 4   tipo_cliente_alto   1000 non-null   float64
 5   tipo_cliente_bajo   1000 non-null   float64
 6   tipo_cliente_medio  1000 non-null   float64
 7   nivel_gasto_alto    1000 non-null   float64
 8   nivel_gasto_bajo    1000 non-null   float64
dtypes: float64(9)
memory usage: 70.4 KB


In [ ]:
# Revisa las estadísticas de una de las variables "escalada"
data_transformada.ingreso_mensual.describe()

,ingreso_mensual
count,1.000000e+03
mean,3.730349e-17
std,1.000500e+00
min,-2.459652e+00
25%,-6.711969e-01
50%,-4.795066e-03
75%,6.781239e-01
max,2.972942e+00


## Uso de `MinMaxScaler`y `OrdinalEncoder`

Acá se aplica `OrdinalEncoder` dado que las variables categóricas disponibles son `ordinales`

In [ ]:
# Define el pipeline para las variables cuantitativas
pipeline_numerico = Pipeline(
    steps=[
        ("escalado", MinMaxScaler())
    ]
)

# Define el pipeline para las variables cualitativas
pipeline_categorico = Pipeline(
    steps=[
        # Acá está lo que había quedado pendiente la semana anterior
        ("encoder", OrdinalEncoder(categories=[
                ["bajo", "medio", "alto"],   # nivel_cliente
                ["bajo", "alto"]           # nivel_gasto
    ]))
    ]
)

# Integra ambos pipelines
preprocesador = ColumnTransformer(
    transformers=[
        ("num", pipeline_numerico, numeric_features),
        ("cat", pipeline_categorico, categorical_features)
    ]
)

# Define el pipeline de transformación
pipeline_transformacion = Pipeline(
    steps=[
        ("transformacion", preprocesador)
    ]
)

In [ ]:
# Aplica la transformación
data_transformada = pipeline_transformacion.fit_transform(data)

In [ ]:
# Crea el Dataframe con los datos transformados
data_transformada = pd.DataFrame(
    pipeline_transformacion.fit_transform(data),
    columns=pipeline_transformacion.named_steps["transformacion"].get_feature_names_out()
)

# Setea el nombre de las columnas
data_transformada.columns = data_transformada.columns.str.replace("num__", "")
data_transformada.columns = data_transformada.columns.str.replace("cat__", "")
data_transformada[numeric_features] = data_transformada[numeric_features].apply(pd.to_numeric)

In [ ]:
data_transformada.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   edad             1000 non-null   float64
 1   ingreso_mensual  1000 non-null   float64
 2   gasto_mensual    1000 non-null   float64
 3   ratio_gasto      1000 non-null   float64
 4   tipo_cliente     1000 non-null   float64
 5   nivel_gasto      1000 non-null   float64
dtypes: float64(6)
memory usage: 47.0 KB


In [ ]:
# Revisa las estadísticas de una de las variables "escalada"
data_transformada.ingreso_mensual.describe()

,ingreso_mensual
count,1000.000000
mean,0.452758
std,0.184166
min,0.000000
25%,0.329208
50%,0.451876
75%,0.577583
max,1.000000


In [ ]:
data_transformada.nivel_gasto.value_counts()

,count
nivel_gasto,
0.0,723
1.0,277


In [ ]:
data.nivel_gasto.value_counts()

,count
nivel_gasto,
bajo,723
alto,277


In [ ]:
data_transformada.tipo_cliente.value_counts()

,count
tipo_cliente,
0.0,400
1.0,396
2.0,204


In [ ]:
data.tipo_cliente.value_counts()

,count
tipo_cliente,
bajo,400
medio,396
alto,204
